In [0]:
CREATE OR REPLACE TEMP VIEW projects_raw AS
SELECT * FROM VALUES
    (101, 'Water Network North', '2026-01-15', '2026-10-30', 1500000, 'Active'),
    (102, 'Drainage Upgrade West', '2026-02-01', '2026-08-15', 900000, 'Completed'),
    (103, 'Pipeline Rehabilitation', '2026-03-10', '2026-12-20', 2200000, 'Active'),
    (104, 'Treatment Plant Expansion', '2026-01-20', '2026-11-30', 3500000, 'Delayed'),
    (105, 'Stormwater Project South', '2026-04-05', '2026-09-30', 750000, 'Active')
AS t(
    project_id,
    project_name,
    start_date,
    planned_end_date,
    budget,
    status
);


In [0]:
CREATE OR REPLACE TEMP VIEW expenses_raw AS
SELECT * FROM VALUES
    (1001, 101, '2026-01-20', 'Materials', 120000),
    (1002, 101, '2026-02-15', 'Labor', 180000),
    (1003, 101, '2026-03-10', 'Equipment', 90000),
    (1004, 102, '2026-02-20', 'Materials', 150000),
    (1005, 102, '2026-04-15', 'Labor', 200000),
    (1006, 102, '2026-06-10', 'Equipment', 80000),
    (1007, 103, '2026-03-20', 'Materials', 300000),
    (1008, 103, '2026-05-12', 'Labor', 250000),
    (1009, 103, '2026-07-05', 'Equipment', 180000),
    (1010, 104, '2026-02-10', 'Materials', 400000),
    (1011, 104, '2026-04-22', 'Labor', 500000),
    (1012, 104, '2026-07-15', 'Equipment', 300000),
    (1013, 105, '2026-04-20', 'Materials', 100000),
    (1014, 105, '2026-06-18', 'Labor', 120000)
AS t(
    expense_id,
    project_id,
    expense_date,
    category,
    amount
);

In [0]:
CREATE OR REPLACE TEMP VIEW projects_budget_ut AS

WITH projects_expenses AS (
    SELECT
        p.project_id,
        p.project_name,
        p.budget,
        e.amount
    FROM projects_raw p
    LEFT JOIN expenses_raw e
        ON p.project_id = e.project_id
),

projects_cost AS (
    SELECT
        project_id,
        project_name,
        budget,
        COALESCE(SUM(amount), 0) AS actual_cost
    FROM projects_expenses
    GROUP BY
        project_id,
        project_name,
        budget
)

SELECT
    project_id,
    project_name,
    budget,
    actual_cost,
    budget - actual_cost AS remaining_budget,
    ROUND((actual_cost / budget) * 100, 2) AS budget_utilization_pct
FROM projects_cost;

SELECT * FROM projects_budget_ut

In [0]:
CREATE OR REPLACE TEMP VIEW cumulative_expenses AS

WITH expenses_total_monthly AS (
    SELECT
        DATE_TRUNC ('MONTH',expense_date) AS month,
        SUM (amount) AS total_expenditure
    FROM expenses_raw
    GROUP BY DATE_TRUNC ('MONTH',expense_date)
)

SELECT
    month,
    total_expenditure,
    SUM (total_expenditure) OVER (
            ORDER BY month ASC
    ) AS cumulative_expenditure
FROM expenses_total_monthly;

SELECT * FROM cumulative_expenses

In [0]:
CREATE OR REPLACE TEMP VIEW projects_expenses_cum AS

WITH projects_expenses_all AS (
    SELECT
        p.project_id,
        p.project_name,
        e.amount,
        DATE_TRUNC ('MONTH',e.expense_date) AS month
    FROM projects_raw p
    LEFT JOIN expenses_raw e
        ON p.project_id = e.project_id
),

projects_expenses_monthly AS (
    SELECT
        month,
        project_id,
        project_name,
        SUM (amount) AS monthly_expenditure
    FROM projects_expenses_all
    GROUP BY month, project_id, project_name
)

SELECT
    month,
    project_id,
    project_name,
    monthly_expenditure,
    SUM (monthly_expenditure) OVER (
        PARTITION BY project_id
        ORDER BY month ASC
    ) AS cumulative_project_expenditure
FROM projects_expenses_monthly;

SELECT * FROM projects_expenses_cum

In [0]:
CREATE OR REPLACE TEMP VIEW highest_expenses AS
SELECT
    project_id,
    project_name,
    budget,
    actual_cost,
    remaining_budget,
    budget_utilization_pct
FROM projects_budget_ut
WHERE budget_utilization_pct > 50
ORDER BY budget_utilization_pct DESC;

SELECT * FROM highest_expenses

In [0]:
CREATE OR REPLACE TEMP VIEW projects_currently_state AS
SELECT
    project_id,
    project_name,
    planned_end_date,
    status,
    DATEDIFF(planned_end_date, '2026-09-27') AS days_to_deadline,
    CASE
        WHEN status = 'Completed'
            THEN 'Completed'
        WHEN planned_end_date < '2026-09-27'
            THEN 'Overdue'
        ELSE 'On Track'
    END AS schedule_status
FROM projects_raw;

SELECT * FROM projects_currently_state 

In [0]:
CREATE OR REPLACE TEMP VIEW project_performance AS
SELECT
    p.project_id,
    p.project_name,
    p.budget,
    p.actual_cost,
    p.remaining_budget,
    p.budget_utilization_pct,
    s.planned_end_date,
    s.status,
    s.days_to_deadline,
    s.schedule_status
FROM projects_budget_ut p
INNER JOIN projects_currently_state s
    ON p.project_id = s.project_id
    WHERE p.budget_utilization_pct > 50 AND s.schedule_status ='Overdue';

SELECT * FROM project_performance

In [0]:
CREATE OR REPLACE TEMP VIEW contractors_raw AS
SELECT * FROM VALUES
    (201, 'AquaBuild S.A.S.', 'Construction'),
    (202, 'InfraWorks Colombia', 'Infrastructure'),
    (203, 'HydroProjects S.A.S.', 'Water Resources'),
    (204, 'UrbanFlow Ltda.', 'Infrastructure')
AS t(
    contractor_id,
    contractor_name,
    contractor_type
);

In [0]:
CREATE OR REPLACE TEMP VIEW project_contractors_raw AS
SELECT * FROM VALUES
    (101, 201),
    (102, 202),
    (103, 203),
    (104, 201),
    (105, 204)
AS t(
    project_id,
    contractor_id
);

In [0]:
CREATE OR REPLACE TEMP VIEW contractor_performance AS

WITH project_and_contractors AS (
    SELECT
        c.contractor_id,
        c.contractor_name,
        c.contractor_type,
        p.project_id
    FROM contractors_raw c
    JOIN project_contractors_raw p
        ON c.contractor_id = p.contractor_id
),

projects_contractors_expenses AS (
    SELECT
        o.contractor_id,
        o.contractor_name,
        o.contractor_type,
        o.project_id,
        u.budget,
        u.actual_cost
    FROM project_and_contractors o
    JOIN projects_budget_ut u
        ON o.project_id = u.project_id
)

SELECT
    contractor_id,
    contractor_name,
    contractor_type,
    COUNT (DISTINCT project_id) AS number_of_projects,
    SUM (budget) AS total_budget,
    SUM (actual_cost) AS total_actual_cost,
    ROUND ((SUM (actual_cost)/SUM (budget)*100),2) AS budget_utilization_pct
FROM projects_contractors_expenses
GROUP BY contractor_id, contractor_name, contractor_type;

SELECT * FROM contractor_performance

In [0]:
CREATE OR REPLACE TEMP VIEW contractors_risk_projects AS

WITH projects_and_contractors AS (
    SELECT
        p.contractor_id,
        p.contractor_name,
        p.number_of_projects AS total_projects,
        r.project_id
    FROM contractor_performance p
    JOIN project_contractors_raw r
        ON p.contractor_id = r.contractor_id
),

projects_performance_risk AS (
    SELECT
        c.contractor_id,
        c.contractor_name,
        c.total_projects,
        p.project_id AS risk_project_id
    FROM projects_and_contractors c
    LEFT JOIN project_performance p
        ON c.project_id = p.project_id
)

SELECT
    contractor_id,
    contractor_name,
    total_projects,
    COUNT(DISTINCT risk_project_id) AS at_risk_projects,
    ROUND(
        COUNT(DISTINCT risk_project_id) * 100.0 / total_projects,
        2
    ) AS risk_project_pct
FROM projects_performance_risk
GROUP BY contractor_id, contractor_name, total_projects;

SELECT * FROM contractors_risk_projects

In [0]:
CREATE OR REPLACE TEMP VIEW contractor_cumulative_expenses AS

WITH project_and_contractors AS (
    SELECT
        p.contractor_id,
        p.contractor_name,
        c.project_id
    FROM contractors_raw p
    JOIN project_contractors_raw c
        ON p.contractor_id = c.contractor_id
),

contractors_all_expenses AS (
    SELECT
        a.contractor_id,
        a.contractor_name,
        a.project_id,
        DATE_TRUNC ('MONTH',e.expense_date) AS month,
        e.amount
    FROM project_and_contractors a
    JOIN expenses_raw e
        ON a.project_id = e.project_id
),

contractor_expenses_monthly AS (
    SELECT
        contractor_id,
        contractor_name,
        month,
        SUM (amount) AS monthly_expenditure
    FROM contractors_all_expenses
    GROUP BY contractor_id, contractor_name, month
)

SELECT 
    contractor_id,
    contractor_name,
    month,
    monthly_expenditure,
    SUM (monthly_expenditure) OVER (
        PARTITION BY contractor_id
        ORDER BY month ASC
    ) AS cumulative_expenditure
FROM contractor_expenses_monthly;

SELECT * FROM contractor_cumulative_expenses

In [0]:
CREATE OR REPLACE TEMP VIEW expense_quality AS

WITH expenses_project AS (
    SELECT
        e.expense_id,
        e.project_id,
        p.project_id AS project_id_p,
        e.expense_date,
        e.category,
        e.amount
    FROM expenses_raw e
    LEFT JOIN projects_raw p
        ON e.project_id = p.project_id
)

SELECT
    expense_id,
    project_id,
    expense_date,
    category,
    amount,
    CASE
        WHEN amount > 0 AND project_id_p IS NOT NULL AND expense_date IS NOT NULL THEN 'Valid'
        ELSE 'Rejected'
    END AS data_quality_status
FROM expenses_project;

SELECT * FROM expense_quality

In [0]:
CREATE OR REPLACE TEMP VIEW expense_quality_summary AS
SELECT
    COUNT (*) AS total_expenses,
    SUM (
        CASE
            WHEN data_quality_status = 'Valid' THEN 1
            ELSE 0
        END
    ) AS valid_expenses,
    SUM (
        CASE
            WHEN data_quality_status = 'Rejected' THEN 1
            ELSE 0
        END
    ) AS rejected_expenses,
    ROUND(SUM (
        CASE
            WHEN data_quality_status = 'Rejected' THEN 1
            ELSE 0
        END
    )*100 / NULLIF(COUNT (*),0), 2) AS rejected_percentage
FROM expense_quality;

SELECT * FROM expense_quality_summary

In [0]:
CREATE OR REPLACE TEMP VIEW clean_expenses AS
SELECT
    expense_id,
    project_id,
    expense_date,
    category,
    amount
FROM expense_quality
WHERE data_quality_status = 'Valid';

SELECT * FROM clean_expenses

In [0]:
CREATE OR REPLACE TEMP VIEW projects_budget_clean AS

WITH projects_expenses AS (
    SELECT
        p.project_id,
        p.project_name,
        p.budget,
        e.amount
    FROM projects_raw p
    LEFT JOIN clean_expenses e
        ON p.project_id = e.project_id
),

projects_cost AS (
    SELECT
        project_id,
        project_name,
        budget,
        COALESCE(SUM(amount), 0) AS actual_cost
    FROM projects_expenses
    GROUP BY
        project_id,
        project_name,
        budget
)

SELECT
    project_id,
    project_name,
    budget,
    actual_cost,
    budget - actual_cost AS remaining_budget,
    ROUND(actual_cost / NULLIF(budget,0) * 100, 2) AS budget_utilization_pct
FROM projects_cost;

SELECT * FROM projects_budget_clean

In [0]:
CREATE OR REPLACE TEMP VIEW cumulative_expenses_clean AS

WITH expenses_total_monthly AS (
    SELECT
        DATE_TRUNC ('MONTH',expense_date) AS month,
        SUM (amount) AS total_expenditure
    FROM clean_expenses
    GROUP BY DATE_TRUNC ('MONTH',expense_date)
)

SELECT
    month,
    total_expenditure,
    SUM (total_expenditure) OVER (
            ORDER BY month ASC
    ) AS cumulative_expenditure
FROM expenses_total_monthly;

SELECT * FROM cumulative_expenses_clean

In [0]:
CREATE OR REPLACE TEMP VIEW projects_performance_clean AS
SELECT
    b.project_id,
    b.project_name,
    b.budget,
    b.actual_cost,
    b.remaining_budget,
    b.budget_utilization_pct,
    s.planned_end_date,
    s.status,
    s.days_to_deadline,
    s.schedule_status
FROM projects_budget_clean b
LEFT JOIN projects_currently_state s
    ON b.project_id = s.project_id;

SELECT * FROM projects_performance_clean

In [0]:
CREATE OR REPLACE TEMP VIEW contractor_performance_clean AS

WITH project_and_contractors AS (
    SELECT
        c.contractor_id,
        c.contractor_name,
        c.contractor_type,
        p.project_id
    FROM contractors_raw c
    JOIN project_contractors_raw p
        ON c.contractor_id = p.contractor_id
),

projects_contractors_expenses AS (
    SELECT
        o.contractor_id,
        o.contractor_name,
        o.contractor_type,
        o.project_id,
        c.budget,
        c.actual_cost
    FROM project_and_contractors o
    JOIN projects_budget_clean c
        ON o.project_id = c.project_id
)

SELECT
    contractor_id,
    contractor_name,
    contractor_type,
    COUNT (DISTINCT project_id) AS number_of_projects,
    SUM (budget) AS total_budget,
    SUM (actual_cost) AS total_actual_cost,
    ROUND ((SUM (actual_cost)/SUM (budget)*100),2) AS budget_utilization_pct
FROM projects_contractors_expenses
GROUP BY contractor_id, contractor_name, contractor_type;

SELECT * FROM contractor_performance_clean

In [0]:
CREATE OR REPLACE TEMP VIEW expenses_incremental AS
SELECT * FROM VALUES
    (1002, 101, '2026-09-28', 'Labor', 185000),
    (1015, 103, '2026-09-28', 'Materials', 95000)
AS t(
    expense_id,
    project_id,
    expense_date,
    category,
    amount
);

SELECT *
FROM expenses_incremental;

In [0]:
CREATE OR REPLACE TABLE expenses_target
USING DELTA
AS
SELECT *
FROM expenses_raw;

In [0]:
MERGE INTO expenses_target AS target
USING expenses_incremental AS source
ON target.expense_id = source.expense_id

WHEN MATCHED THEN
    UPDATE SET
        target.project_id = source.project_id,
        target.expense_date = source.expense_date,
        target.category = source.category,
        target.amount = source.amount

WHEN NOT MATCHED THEN
    INSERT (
        expense_id,
        project_id,
        expense_date,
        category,
        amount
    )
    VALUES (
        source.expense_id,
        source.project_id,
        source.expense_date,
        source.category,
        source.amount
    );

In [0]:
SELECT *
FROM expenses_target
ORDER BY expense_id;

In [0]:
CREATE OR REPLACE TEMP VIEW project_quality_checks AS

SELECT
    'Duplicate projects' AS check_name,
    COUNT(*) AS failed_records,
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'REVIEW'
    END AS check_status
FROM (
    SELECT project_id
    FROM projects_performance_clean
    GROUP BY project_id
    HAVING COUNT(*) > 1
) duplicates

UNION ALL

SELECT
    'Negative budget' AS check_name,
    SUM(
        CASE
            WHEN budget < 0 THEN 1
            ELSE 0
        END
    ) AS failed_records,
    CASE
        WHEN SUM(CASE WHEN budget < 0 THEN 1 ELSE 0 END) = 0
            THEN 'PASS'
        ELSE 'REVIEW'
    END AS check_status
FROM projects_performance_clean

UNION ALL

SELECT
    'Over budget' AS check_name,
    SUM(
        CASE
            WHEN actual_cost > budget THEN 1
            ELSE 0
        END
    ) AS failed_records,
    CASE
        WHEN SUM(CASE WHEN actual_cost > budget THEN 1 ELSE 0 END) = 0
            THEN 'PASS'
        ELSE 'REVIEW'
    END AS check_status
FROM projects_performance_clean;

SELECT *
FROM project_quality_checks;

In [0]:
CREATE OR REPLACE TEMP VIEW dim_project AS
SELECT
    r.project_id,
    r.project_name,
    c.contractor_id
FROM projects_raw r
LEFT JOIN project_contractors_raw c
    ON r.project_id = c.project_id;

SELECT * FROM dim_project

In [0]:
CREATE OR REPLACE TEMP VIEW dim_contractor AS
SELECT
    contractor_id,
    contractor_name,
    contractor_type
FROM contractors_raw;

SELECT * FROM dim_contractor

In [0]:
CREATE OR REPLACE TEMP VIEW dim_date AS

WITH calendar AS (
    SELECT 
    EXPLODE(
        SEQUENCE(
            DATE('2026-01-01'),
            DATE('2026-12-31'),
            INTERVAL 1 DAY
        )
    ) AS date
)

SELECT
    date,
    YEAR (date) AS year,
    QUARTER (date) AS quarter,
    MONTH (date) AS month_number,
    MONTHNAME (date) AS month_name,
    DATE_FORMAT(date, 'yyyy-MM') AS year_month
FROM calendar;

SELECT * FROM dim_date

In [0]:
CREATE OR REPLACE TEMP VIEW fact_expenses AS
SELECT
    expense_id,
    project_id,
    expense_date,
    category,
    amount
FROM clean_expenses;

SELECT * FROM fact_expenses

In [0]:
CREATE OR REPLACE TEMP VIEW fact_project_performance AS
SELECT
    project_id,
    budget,
    actual_cost,
    remaining_budget,
    budget_utilization_pct,
    planned_end_date,
    days_to_deadline,
    status,
    schedule_status
FROM projects_performance_clean;

SELECT * FROM fact_project_performance

In [0]:
SELECT
    'fact_expenses' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT expense_id) AS distinct_keys
FROM fact_expenses

UNION ALL

SELECT
    'fact_project_performance' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT project_id) AS distinct_keys
FROM fact_project_performance;

In [0]:
SELECT
    'dim_project' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT project_id) AS distinct_keys
FROM dim_project

UNION ALL

SELECT
    'dim_contractor',
    COUNT(*),
    COUNT(DISTINCT contractor_id)
FROM dim_contractor

UNION ALL

SELECT
    'dim_date',
    COUNT(*),
    COUNT(DISTINCT date)
FROM dim_date;


In [0]:
SELECT
    current_catalog() AS current_catalog,
    current_schema() AS current_schema;

In [0]:
SHOW SCHEMAS IN workspace;

In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.infrastructure_analytics;

In [0]:
SHOW SCHEMAS IN workspace;

In [0]:
CREATE OR REPLACE TABLE workspace.infrastructure_analytics.dim_project
USING DELTA
AS
SELECT *
FROM dim_project;

In [0]:
SELECT *
FROM workspace.infrastructure_analytics.dim_project;

In [0]:
CREATE OR REPLACE TABLE workspace.infrastructure_analytics.dim_contractor
USING DELTA
AS
SELECT *
FROM dim_contractor;

In [0]:
CREATE OR REPLACE TABLE workspace.infrastructure_analytics.dim_date
USING DELTA
AS
SELECT *
FROM dim_date;

In [0]:
CREATE OR REPLACE TABLE workspace.infrastructure_analytics.fact_expenses
USING DELTA
AS
SELECT *
FROM fact_expenses;

In [0]:
CREATE OR REPLACE TABLE workspace.infrastructure_analytics.fact_project_performance
USING DELTA
AS
SELECT *
FROM fact_project_performance;

In [0]:
SELECT 'dim_project' AS table_name, COUNT(*) AS total_rows
FROM workspace.infrastructure_analytics.dim_project

UNION ALL

SELECT 'dim_contractor', COUNT(*)
FROM workspace.infrastructure_analytics.dim_contractor

UNION ALL

SELECT 'dim_date', COUNT(*)
FROM workspace.infrastructure_analytics.dim_date

UNION ALL

SELECT 'fact_expenses', COUNT(*)
FROM workspace.infrastructure_analytics.fact_expenses

UNION ALL

SELECT 'fact_project_performance', COUNT(*)
FROM workspace.infrastructure_analytics.fact_project_performance;